<h1 style="color: #1E90FF; font-size: 2.5em; font-weight: bold;">
    FIFA 21 Player Market Value — ML Regression Models with Full Pipeline
</h1>

### <font color='#1E90FF'>**Table of Contents**</font> <a class="anchor" id='toc'></a>

- [1. Setup & Load Raw Data](#1)
- [2. Data Engineering Pipeline (Preprocessing)](#2)
    - [2.1. Raw Cleaning with Spark SQL Transformers](#2_1)
    - [2.2. Custom Spark ML Transformers](#2_2)
- [3. Feature Selection & ML Preprocessing](#3)
- [4. Train/Test Split](#4)
- [5. Full End-to-End Pipeline](#5)
    - [5.1. Linear Regression](#5_1)
    - [5.2. Random Forest](#5_2)
    - [5.3. Gradient Boosted Trees (GBT)](#5_3)
- [6. Hyperparameter Tuning with CrossValidator](#6)
    - [6.1. Tuning Random Forest](#6_1)
    - [6.2. Tuning GBT](#6_2)
- [7. Model Comparison](#7)
- [8. Overfitting Analysis](#8)
- [9. Feature Importance](#9)

<a class="anchor" id="1"></a>

# **1. Setup & Load Raw Data**

[Back to TOC](#toc)

We load the **raw** FIFA 21 CSV directly here — no pre-cleaned parquet needed. All preprocessing is encapsulated inside the Pipeline, as required by the **Pipelines & Data Engineering** criterion of the project brief. This means the pipeline can be applied to any new raw data without manual steps.

In [1]:
!pip install pyspark plotly "pandas>=2.2.0" "nbformat>=4.2.0"

In [2]:
# Install Java 17 (Required for Spark)
!sudo apt-get update -q
!sudo apt-get install -y -q openjdk-17-jdk-headless
!java -version

Hit:1 https://packages.cloud.google.com/apt cloud-sdk InRelease
Get:2 https://download.docker.com/linux/ubuntu noble InRelease [48.5 kB]
Hit:3 https://cli.github.com/packages stable InRelease
Get:4 https://nvidia.github.io/libnvidia-container/stable/deb/amd64  InRelease [1477 B]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease [1581 B]
Get:6 https://archive.ubuntu.com/ubuntu noble InRelease [256 kB]
Hit:7 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease
Get:8 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:9 https://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:10 https://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:11 https://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:12 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:13 https://nvidia.github.io/libnvidia-container/stable/deb/amd64  Packages [26

In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, log1p, when, expm1, regexp_replace, split, trim,
    size, regexp_extract, to_date, year, length
)
from pyspark.ml import Pipeline, Transformer
from pyspark.ml.param.shared import HasInputCol, HasOutputCol
from pyspark.ml.util import DefaultParamsReadable, DefaultParamsWritable
from pyspark.ml.feature import (
    VectorAssembler, StandardScaler, StringIndexer,
    OneHotEncoder, SQLTransformer, Imputer
)
from pyspark.ml.regression import (
    LinearRegression, RandomForestRegressor, GBTRegressor
)
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

spark = SparkSession.builder \
    .master("local[4]") \
    .appName("FIFA21 ML Pipeline — Full") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

print("Spark Session ready!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/03 20:24:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Session ready!


In [4]:
# Load raw CSV — same as nb1
raw = spark.read.csv('./fifa21 raw data v2.csv', header=True, multiLine=True)
raw.cache()

print(f"Raw rows   : {raw.count()}")
print(f"Raw columns: {len(raw.columns)}")

26/06/03 20:24:40 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Raw rows   : 18979
Raw columns: 77


<a class="anchor" id="2"></a>

# **2. Data Engineering Pipeline (Preprocessing)**

[Back to TOC](#toc)

The project brief explicitly requires demonstrating **Pipelines & Data Engineering**. Instead of cleaning the data manually before ML (as done in `01_EDA_and_preprocessing.ipynb`), we encapsulate all cleaning and feature engineering steps into **Spark ML Pipeline stages**.

This approach has the following big-data advantages:
- **Reproducibility**: the same pipeline object transforms any new batch of raw data identically.
- **No data leakage**: the pipeline is fit only on training data; `StringIndexer`, `Imputer`, and `StandardScaler` learn their statistics exclusively from the training fold, which is critical for correct cross-validation.
- **Scalability**: all transformations are lazy and distributed — Spark only materialises data when an action triggers it.

<a class="anchor" id="2_1"></a>

## **2.1. Raw Cleaning with SQLTransformer**

[Back to TOC](#toc)

`SQLTransformer` is a built-in Spark ML stage that applies a SQL `SELECT` statement to the incoming DataFrame. We use it to perform the same column renames and basic casts that were done manually in nb1, but now as a reusable pipeline stage.

> **⚠️ Big-data safety note**: `SQLTransformer` pushes all operations down to the Catalyst query planner. Every expression used here is partition-safe and does not require a shuffle or global sort.

In [5]:
# Stage 1 — SQLTransformer: rename messy columns, strip symbols, cast numerics
# This mirrors the column renames and type casts from nb1 section 6.1
# NOTE: backtick quoting is required for column names with spaces or special characters.

sql_clean = SQLTransformer(statement="""
SELECT
    ID,
    Name,
    LongName,
    Nationality,
    CAST(Age AS INT)                                                         AS Age,
    TRIM(REGEXP_REPLACE(Club, '[\\n]', ''))                                  AS Club,
    Contract,
    Positions,
    CAST(
        CASE WHEN Height LIKE "%'%"
             THEN (CAST(SPLIT(Height, "'")[0] AS FLOAT) * 30.48)
                + (CAST(REGEXP_REPLACE(SPLIT(Height, "'")[1], '"', '') AS FLOAT) * 2.54)
             ELSE CAST(REGEXP_REPLACE(Height, 'cm', '') AS FLOAT)
        END AS INT)                                                          AS Height,
    CAST(
        CASE WHEN Weight LIKE '%lbs'
             THEN CAST(REGEXP_REPLACE(Weight, 'lbs', '') AS FLOAT) * 0.453592
             ELSE CAST(REGEXP_REPLACE(Weight, 'kg', '') AS FLOAT)
        END AS INT)                                                          AS Weight,
    `Preferred Foot`                                                         AS Preferred_Foot,
    Joined,
    `Loan Date End`,
    CAST(`↓OVA` AS INT)                                                         AS OVA,
    CAST(POT AS INT)                                                         AS POT,
    CAST(BOV AS INT)                                                         AS BOV,
    `Best Position`                                                          AS Best_Position,
    CAST(
        CASE WHEN Value LIKE '%M' THEN CAST(REGEXP_REPLACE(Value, '[€M]', '') AS FLOAT) * 1000000
             WHEN Value LIKE '%K' THEN CAST(REGEXP_REPLACE(Value, '[€K]', '') AS FLOAT) * 1000
             ELSE CAST(REGEXP_REPLACE(Value, '[€]', '') AS FLOAT)
        END AS INT)                                                          AS Value,
    CAST(
        CASE WHEN Wage LIKE '%K' THEN CAST(REGEXP_REPLACE(Wage, '[€K]', '') AS FLOAT) * 1000
             ELSE CAST(REGEXP_REPLACE(Wage, '[€]', '') AS FLOAT)
        END AS INT)                                                          AS Wage,
    CAST(
        CASE WHEN `Release Clause` LIKE '%M' THEN CAST(REGEXP_REPLACE(`Release Clause`, '[€M]', '') AS FLOAT) * 1000000
             WHEN `Release Clause` LIKE '%K' THEN CAST(REGEXP_REPLACE(`Release Clause`, '[€K]', '') AS FLOAT) * 1000
             ELSE CAST(REGEXP_REPLACE(`Release Clause`, '[€]', '') AS FLOAT)
        END AS INT)                                                          AS Release_Clause,
    CAST(REGEXP_REPLACE(`W/F`,  '[^0-9]', '') AS INT)                       AS WF,
    CAST(REGEXP_REPLACE(`SM`,   '[^0-9]', '') AS INT)                       AS SM,
    `A/W`                                                                    AS AW,
    `D/W`                                                                    AS DW,
    CAST(REGEXP_REPLACE(IR,     '[^0-9]', '') AS INT)                       AS IR,
    CAST(Attacking          AS INT) AS Attacking,
    CAST(Crossing           AS INT) AS Crossing,
    CAST(Finishing          AS INT) AS Finishing,
    CAST(`Heading Accuracy` AS INT) AS Heading_Accuracy,
    CAST(`Short Passing`    AS INT) AS Short_Passing,
    CAST(Volleys            AS INT) AS Volleys,
    CAST(Skill              AS INT) AS Skill,
    CAST(Dribbling          AS INT) AS Dribbling,
    CAST(Curve              AS INT) AS Curve,
    CAST(`FK Accuracy`      AS INT) AS FK_Accuracy,
    CAST(`Long Passing`     AS INT) AS Long_Passing,
    CAST(`Ball Control`     AS INT) AS Ball_Control,
    CAST(Movement           AS INT) AS Movement,
    CAST(Acceleration       AS INT) AS Acceleration,
    CAST(`Sprint Speed`     AS INT) AS Sprint_Speed,
    CAST(Agility            AS INT) AS Agility,
    CAST(Reactions          AS INT) AS Reactions,
    CAST(Balance            AS INT) AS Balance,
    CAST(Power              AS INT) AS Power,
    CAST(`Shot Power`       AS INT) AS Shot_Power,
    CAST(Jumping            AS INT) AS Jumping,
    CAST(Stamina            AS INT) AS Stamina,
    CAST(Strength           AS INT) AS Strength,
    CAST(`Long Shots`       AS INT) AS Long_Shots,
    CAST(Mentality          AS INT) AS Mentality,
    CAST(Aggression         AS INT) AS Aggression,
    CAST(Interceptions      AS INT) AS Interceptions,
    CAST(Positioning        AS INT) AS Positioning,
    CAST(Vision             AS INT) AS Vision,
    CAST(Penalties          AS INT) AS Penalties,
    CAST(Composure          AS INT) AS Composure,
    CAST(Defending          AS INT) AS Defending,
    CAST(Marking            AS INT) AS Marking,
    CAST(`Standing Tackle`  AS INT) AS Standing_Tackle,
    CAST(`Sliding Tackle`   AS INT) AS Sliding_Tackle,
    CAST(Goalkeeping        AS INT) AS Goalkeeping,
    CAST(`GK Diving`        AS INT) AS GK_Diving,
    CAST(`GK Handling`      AS INT) AS GK_Handling,
    CAST(`GK Kicking`       AS INT) AS GK_Kicking,
    CAST(`GK Positioning`   AS INT) AS GK_Positioning,
    CAST(`GK Reflexes`      AS INT) AS GK_Reflexes,
    CAST(`Total Stats`      AS INT) AS Total_Stats,
    CAST(`Base Stats`       AS INT) AS Base_Stats,
    CAST(PAC AS INT) AS PAC,
    CAST(SHO AS INT) AS SHO,
    CAST(PAS AS INT) AS PAS,
    CAST(DRI AS INT) AS DRI,
    CAST(DEF AS INT) AS DEF,
    CAST(PHY AS INT) AS PHY
FROM __THIS__
""")

print("SQLTransformer stage defined.")

SQLTransformer stage defined.


<a class="anchor" id="2_2"></a>

## **2.2. Custom Spark ML Transformers (Feature Engineering)**

[Back to TOC](#toc)

For feature engineering steps that cannot be expressed in a single SQL `SELECT` (e.g., multi-step contract parsing, position counting), we implement **custom `Transformer` subclasses**. Each subclass follows the Spark ML `Transformer` API and is therefore a fully composable, serialisable Pipeline stage.

This is a key demonstration of **data engineering skills**: building reusable, production-quality transformation components rather than ad-hoc scripts.

In [6]:
from pyspark import keyword_only
from pyspark.ml.param import Params

class ContractFeatureEngineer(Transformer, DefaultParamsReadable, DefaultParamsWritable):
    """
    Custom Spark ML Transformer that replicates the contract feature engineering
    from nb1 section 8:
      - Contract_Status  : 'On Loan' | 'Free' | 'Permanent'
      - Contract_Start_Year : year from 'Joined' column (null for free agents)
      - Contract_End_Year   : end year extracted from Contract string

    Big-data safe: all operations are column-level transformations with no
    collect() or toPandas() calls. Runs fully distributed.
    """

    @keyword_only
    def __init__(self):
        super().__init__()

    def _transform(self, df):
        df = df.withColumn("Contract_Status",
            when(col("Contract").contains("On Loan"), "On Loan")
            .when(col("Contract").contains("Free"), "Free")
            .otherwise("Permanent")
        )
        df = df.withColumn("Contract_Start_Year",
            when(col("Contract_Status") == "Free", None)
            .otherwise(year(to_date(col("Joined"), "MMM d, yyyy")))
        )
        df = df.withColumn("Contract_End_Year",
            when(col("Contract_Status") == "Permanent",
                 trim(split(col("Contract"), "~")[1]).cast("int"))
            .when(col("Contract_Status") == "On Loan",
                 regexp_extract(col("Contract"), r"(\d{4})", 1).cast("int"))
            .otherwise(None)
        )
        # Drop raw columns that are no longer needed
        df = df.drop("Contract", "Joined", "Loan Date End")
        return df


class NPositionsEngineer(Transformer, DefaultParamsReadable, DefaultParamsWritable):
    """
    Custom Transformer that creates 'N_Positions' — the number of positions
    a player can play (from nb1 section 8). Captures player versatility.

    Big-data safe: uses Spark built-in size() and split() — no UDFs.
    """

    @keyword_only
    def __init__(self):
        super().__init__()

    def _transform(self, df):
        return df.withColumn("N_Positions", size(split(col("Positions"), ",")))


class LogTargetTransformer(Transformer, DefaultParamsReadable, DefaultParamsWritable):
    """
    Applies log1p to the 'Value' column to create 'log_Value' (the regression target),
    and filters out players with Value == 0 (free agents with no market value).

    Placing this inside the Pipeline ensures the log-transformation is always
    consistently applied to training and inference data.
    """

    @keyword_only
    def __init__(self):
        super().__init__()

    def _transform(self, df):
        return (
            df.filter(col("Value") > 0)
              .withColumn("log_Value", log1p(col("Value")))
        )


print("Custom Transformer classes defined.")

Custom Transformer classes defined.


<a class="anchor" id="3"></a>

# **3. Feature Selection & ML Preprocessing**

[Back to TOC](#toc)

We define the necessary feature groups for our models, integrating all encoding and scaling steps (`StringIndexer`, `OneHotEncoder`, `Imputer`, `VectorAssembler`, `StandardScaler`) directly as Pipeline stages. This ensures they learn exclusively from the training data.

**No data leakage**: `Wage` and `Release_Clause` are excluded — they are derived from `Value` and would be unavailable at prediction time for a new player.

In [7]:
numeric_features = [
    "Age", "OVA", "POT", "BOV",
    "Height", "Weight",
    "Attacking", "Crossing", "Finishing", "Heading_Accuracy",
    "Short_Passing", "Volleys",
    "Skill", "Dribbling", "Curve", "FK_Accuracy",
    "Long_Passing", "Ball_Control",
    "Movement", "Acceleration", "Sprint_Speed", "Agility",
    "Reactions", "Balance",
    "Power", "Shot_Power", "Jumping", "Stamina", "Strength", "Long_Shots",
    "Mentality", "Aggression", "Interceptions", "Positioning",
    "Vision", "Penalties", "Composure",
    "Defending", "Marking", "Standing_Tackle", "Sliding_Tackle",
    "Goalkeeping", "GK_Diving", "GK_Handling", "GK_Kicking",
    "GK_Positioning", "GK_Reflexes",
    "Total_Stats", "Base_Stats",
    "WF", "SM", "IR",
    "N_Positions"
]

categorical_features = ["Preferred_Foot", "Contract_Status", "AW","DW"]

print(f"Numeric features : {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

Numeric features : 53
Categorical features: 4


In [8]:
# --- Categorical encoding stages ---
# StringIndexer learns the mapping from training data only (no leakage)
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_features
]
encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in categorical_features
]
ohe_cols = [f"{c}_ohe" for c in categorical_features]

# --- Imputer for numeric nulls (mean strategy, big-data safe) ---
# Learns mean from training split only — essential for correct cross-validation.
# NOTE: We use Imputer here as a proper Pipeline stage, unlike nb1 where nulls were
# dropped with dropna() outside the pipeline (not big-data safe for CV).
imputer = Imputer(
    inputCols=numeric_features,
    outputCols=[f"{c}_imp" for c in numeric_features],
    strategy="mean"
)
imputed_numeric = [f"{c}_imp" for c in numeric_features]

# --- VectorAssembler ---
assembler = VectorAssembler(
    inputCols=imputed_numeric + ohe_cols,
    outputCol="raw_features",
    handleInvalid="skip"
)

# --- StandardScaler (used only for Linear Regression) ---
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,
    withStd=True
)

print("ML preprocessing stages defined.")

ML preprocessing stages defined.


<a class="anchor" id="4"></a>

# **4. Train / Test Split**

[Back to TOC](#toc)

The split is performed on the **raw** data (before any pipeline is fit), which is the correct approach. The Pipeline's `fit()` will only see training rows, preventing any statistics (e.g., imputation means, scaler parameters) from being computed on test data.

In [9]:
train_raw, test_raw = raw.randomSplit([0.7, 0.3], seed=42)

# Cache splits — each will be accessed multiple times during pipeline.fit()
train_raw.cache()
test_raw.cache()

print(f"Train rows (raw): {train_raw.count()}")
print(f"Test rows  (raw): {test_raw.count()}")

Train rows (raw): 13365
Test rows  (raw): 5614


<a class="anchor" id="5"></a>

# **5. Full End-to-End Pipeline**

[Back to TOC](#toc)

Each model pipeline is constructed as:

```
Raw CSV
  → SQLTransformer      (column renames, symbol stripping, type casts)
  → ContractFeatureEngineer  (Contract_Status, Contract_Start_Year, Contract_End_Year)
  → NPositionsEngineer  (N_Positions)
  → LogTargetTransformer     (filter Value > 0, create log_Value)
  → StringIndexer × 2   (encode Preferred_Foot, Contract_Status)
  → OneHotEncoder × 2
  → Imputer             (fill numeric nulls with training mean)
  → VectorAssembler
  → [StandardScaler]    (only for Linear Regression)
  → ML Model
```

> **Why is this better than nb1's approach?** In nb1, preprocessing was manual and stateful — cleaning steps ran once over the entire dataset before the train/test split. This means test-set statistics could inadvertently influence cleaning decisions. With a Pipeline, `fit()` only observes training data for any learned step.

In [10]:
# Shared preprocessing stages (before the model)
shared_stages = [
    sql_clean,
    ContractFeatureEngineer(),
    NPositionsEngineer(),
    LogTargetTransformer(),
] + indexers + encoders + [imputer, assembler]

print(f"Total shared stages (excl. model): {len(shared_stages)}")

Total shared stages (excl. model): 14


In [11]:
# --- Helper: evaluate a fitted pipeline on a dataset ---
from pyspark.sql.functions import expm1 as spark_expm1

evaluator_rmse = RegressionEvaluator(labelCol="log_Value", predictionCol="prediction", metricName="rmse")
evaluator_r2   = RegressionEvaluator(labelCol="log_Value", predictionCol="prediction", metricName="r2")
evaluator_mae  = RegressionEvaluator(labelCol="log_Value", predictionCol="prediction", metricName="mae")

results = {}  # model name -> metrics dict

def evaluate_model(name, pipeline_model, test_data):
    preds = pipeline_model.transform(test_data)
    rmse_log = evaluator_rmse.evaluate(preds)
    r2       = evaluator_r2.evaluate(preds)
    mae_log  = evaluator_mae.evaluate(preds)
    preds_euro = (
        preds
        .withColumn("Value_Euro",      spark_expm1(col("log_Value")))
        .withColumn("Prediction_Euro", spark_expm1(col("prediction")))
    )
    mae_euro = RegressionEvaluator(
        labelCol="Value_Euro", predictionCol="Prediction_Euro", metricName="mae"
    ).evaluate(preds_euro)
    results[name] = {
        "R²": round(r2, 4),
        "RMSE (log)": round(rmse_log, 4),
        "MAE (log)": round(mae_log, 4),
        "MAE (€)": round(mae_euro, 2)
    }
    print(f"\n{'='*40}")
    print(f"Model : {name}")
    print(f"  R²              : {r2:.4f}")
    print(f"  RMSE (log-scale): {rmse_log:.4f}")
    print(f"  MAE  (log-scale): {mae_log:.4f}")
    print(f"  MAE  (Euros €)  : {mae_euro:,.2f} €")
    return preds

<a class="anchor" id="5_1"></a>

## **5.1. Linear Regression**

[Back to TOC](#toc)

Linear Regression requires feature scaling (`StandardScaler`) because it is sensitive to feature magnitude. This is added only for LR — tree-based models do not need it.

In [12]:
lr = LinearRegression(
    featuresCol="features",
    labelCol="log_Value",
    maxIter=100,
    regParam=0.01,
    elasticNetParam=0.0  # Ridge (L2)
)

# LR pipeline includes the StandardScaler (tree models skip this stage)
pipeline_lr = Pipeline(stages=shared_stages + [scaler, lr])
model_lr    = pipeline_lr.fit(train_raw)

preds_lr = evaluate_model("Linear Regression (baseline)", model_lr, test_raw)

26/06/03 20:24:56 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/06/03 20:24:57 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK



Model : Linear Regression (baseline)
  R²              : 0.9660
  RMSE (log-scale): 0.2291
  MAE  (log-scale): 0.1644
  MAE  (Euros €)  : 607,781.60 €


In [13]:
# Training summary (available from the last stage of the fitted pipeline)
lr_summary = model_lr.stages[-1].summary
print(f"Training RMSE : {lr_summary.rootMeanSquaredError:.4f}")
print(f"Training R²   : {lr_summary.r2:.4f}")

Training RMSE : 0.2318
Training R²   : 0.9652


<a class="anchor" id="5_2"></a>

## **5.2. Random Forest**

[Back to TOC](#toc)

Tree-based models are invariant to feature scaling, so `StandardScaler` is omitted. They use `raw_features` directly from the `VectorAssembler`.

In [14]:
rf = RandomForestRegressor(
    featuresCol="raw_features",
    labelCol="log_Value",
    numTrees=100,
    maxDepth=8,
    seed=42
)

# NOTE: StandardScaler NOT included — tree models don't need scaling
pipeline_rf = Pipeline(stages=shared_stages + [rf])
model_rf    = pipeline_rf.fit(train_raw)

preds_rf = evaluate_model("Random Forest (baseline)", model_rf, test_raw)

26/06/03 20:25:11 WARN DAGScheduler: Broadcasting large task binary with size 1269.9 KiB
26/06/03 20:25:12 WARN DAGScheduler: Broadcasting large task binary with size 1865.8 KiB
26/06/03 20:25:14 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
26/06/03 20:25:16 WARN DAGScheduler: Broadcasting large task binary with size 5.3 MiB
26/06/03 20:25:18 WARN DAGScheduler: Broadcasting large task binary with size 1184.9 KiB



Model : Random Forest (baseline)
  R²              : 0.9883
  RMSE (log-scale): 0.1346
  MAE  (log-scale): 0.0951
  MAE  (Euros €)  : 344,961.98 €


<a class="anchor" id="5_3"></a>

## **5.3. Gradient Boosted Trees (GBT)**

[Back to TOC](#toc)

In [15]:
gbt = GBTRegressor(
    featuresCol="raw_features",
    labelCol="log_Value",
    maxIter=100,
    maxDepth=5,
    stepSize=0.1,
    seed=42
)

pipeline_gbt = Pipeline(stages=shared_stages + [gbt])
model_gbt    = pipeline_gbt.fit(train_raw)

preds_gbt = evaluate_model("GBT (baseline)", model_gbt, test_raw)

26/06/03 20:25:45 WARN DAGScheduler: Broadcasting large task binary with size 1002.2 KiB
26/06/03 20:25:45 WARN DAGScheduler: Broadcasting large task binary with size 1005.0 KiB
26/06/03 20:25:45 WARN DAGScheduler: Broadcasting large task binary with size 1005.5 KiB
26/06/03 20:25:46 WARN DAGScheduler: Broadcasting large task binary with size 1006.5 KiB
26/06/03 20:25:46 WARN DAGScheduler: Broadcasting large task binary with size 1007.2 KiB
26/06/03 20:25:46 WARN DAGScheduler: Broadcasting large task binary with size 1009.5 KiB
26/06/03 20:25:46 WARN DAGScheduler: Broadcasting large task binary with size 1012.3 KiB
26/06/03 20:25:46 WARN DAGScheduler: Broadcasting large task binary with size 1012.8 KiB
26/06/03 20:25:46 WARN DAGScheduler: Broadcasting large task binary with size 1013.8 KiB
26/06/03 20:25:46 WARN DAGScheduler: Broadcasting large task binary with size 1014.5 KiB
26/06/03 20:25:46 WARN DAGScheduler: Broadcasting large task binary with size 1016.8 KiB
26/06/03 20:25:46 WAR


Model : GBT (baseline)
  R²              : 0.9952
  RMSE (log-scale): 0.0859
  MAE  (log-scale): 0.0593
  MAE  (Euros €)  : 214,294.66 €


<a class="anchor" id="6"></a>

# **6. Hyperparameter Tuning with CrossValidator**

[Back to TOC](#toc)

We use 5-fold `CrossValidator` for the two tree models. Crucially, because the full preprocessing is inside the pipeline, **each fold correctly refits the `Imputer` and `StringIndexer` on its own training data** — this is not possible when preprocessing is done outside the pipeline.

> **Big-data note**: `CrossValidator` triggers 5 × N full pipeline fits per parameter combination. This is expensive but fully distributed on a Spark cluster.

<a class="anchor" id="6_1"></a>

## **6.1. Tuning Random Forest**

[Back to TOC](#toc)

In [16]:
rf_tuned = RandomForestRegressor(
    featuresCol="raw_features",
    labelCol="log_Value",
    seed=42
)

pipeline_rf_tune = Pipeline(stages=shared_stages + [rf_tuned])

param_grid_rf = (
    ParamGridBuilder()
    .addGrid(rf_tuned.numTrees,             [50, 100, 200])
    .addGrid(rf_tuned.maxDepth,             [6, 8, 10])
    .addGrid(rf_tuned.minInstancesPerNode,  [1, 5])
    .build()
)

cv_rf = CrossValidator(
    estimator=pipeline_rf_tune,
    estimatorParamMaps=param_grid_rf,
    evaluator=evaluator_rmse,
    numFolds=5,
    seed=42
)

print(f"Fitting RF CrossValidator — {len(param_grid_rf)} param combos × 5 folds ...")
cv_model_rf = cv_rf.fit(train_raw)

best_rf_params = cv_model_rf.bestModel.stages[-1].extractParamMap()
print("\nBest RF parameters:")
for param, val in best_rf_params.items():
    print(f"  {param.name}: {val}")

Fitting RF CrossValidator — 18 param combos × 5 folds ...


26/06/03 20:26:19 WARN DAGScheduler: Broadcasting large task binary with size 1012.2 KiB
26/06/03 20:26:19 WARN DAGScheduler: Broadcasting large task binary with size 1310.7 KiB
26/06/03 20:26:26 WARN DAGScheduler: Broadcasting large task binary with size 1012.2 KiB
26/06/03 20:26:26 WARN DAGScheduler: Broadcasting large task binary with size 1311.8 KiB
26/06/03 20:26:33 WARN DAGScheduler: Broadcasting large task binary with size 1012.2 KiB
26/06/03 20:26:33 WARN DAGScheduler: Broadcasting large task binary with size 1310.7 KiB
26/06/03 20:26:34 WARN DAGScheduler: Broadcasting large task binary with size 1901.3 KiB
26/06/03 20:26:35 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
26/06/03 20:26:42 WARN DAGScheduler: Broadcasting large task binary with size 1012.1 KiB
26/06/03 20:26:42 WARN DAGScheduler: Broadcasting large task binary with size 1311.8 KiB
26/06/03 20:26:43 WARN DAGScheduler: Broadcasting large task binary with size 1897.4 KiB
26/06/03 20:26:44 WARN D


Best RF parameters:
  bootstrap: True
  cacheNodeIds: False
  checkpointInterval: 10
  featureSubsetStrategy: auto
  featuresCol: raw_features
  impurity: variance
  labelCol: log_Value
  leafCol: 
  maxBins: 32
  maxDepth: 10
  maxMemoryInMB: 256
  minInfoGain: 0.0
  minInstancesPerNode: 1
  minWeightFractionPerNode: 0.0
  numTrees: 200
  predictionCol: prediction
  seed: 42
  subsamplingRate: 1.0


In [17]:
preds_rf_tuned = evaluate_model("Random Forest (tuned)", cv_model_rf, test_raw)


Model : Random Forest (tuned)
  R²              : 0.9925
  RMSE (log-scale): 0.1078
  MAE  (log-scale): 0.0746
  MAE  (Euros €)  : 275,777.74 €


<a class="anchor" id="6_2"></a>

## **6.2. Tuning GBT**

[Back to TOC](#toc)

In [18]:
gbt_tuned = GBTRegressor(
    featuresCol="raw_features",
    labelCol="log_Value",
    seed=42
)

pipeline_gbt_tune = Pipeline(stages=shared_stages + [gbt_tuned])

param_grid_gbt = (
    ParamGridBuilder()
    .addGrid(gbt_tuned.maxIter,  [50, 100, 150])
    .addGrid(gbt_tuned.maxDepth, [4, 5, 6])
    .addGrid(gbt_tuned.stepSize, [0.05, 0.1, 0.2])
    .build()
)

cv_gbt = CrossValidator(
    estimator=pipeline_gbt_tune,
    estimatorParamMaps=param_grid_gbt,
    evaluator=evaluator_rmse,
    numFolds=5,
    seed=42
)

print(f"Fitting GBT CrossValidator — {len(param_grid_gbt)} param combos × 5 folds ...")
cv_model_gbt = cv_gbt.fit(train_raw)

best_gbt_params = cv_model_gbt.bestModel.stages[-1].extractParamMap()
print("\nBest GBT parameters:")
for param, val in best_gbt_params.items():
    print(f"  {param.name}: {val}")

Fitting GBT CrossValidator — 27 param combos × 5 folds ...


26/06/03 20:54:38 WARN DAGScheduler: Broadcasting large task binary with size 1001.5 KiB
26/06/03 20:54:38 WARN DAGScheduler: Broadcasting large task binary with size 1001.9 KiB
26/06/03 20:54:38 WARN DAGScheduler: Broadcasting large task binary with size 1002.9 KiB
26/06/03 20:54:38 WARN DAGScheduler: Broadcasting large task binary with size 1003.7 KiB
26/06/03 20:54:38 WARN DAGScheduler: Broadcasting large task binary with size 1006.0 KiB
26/06/03 20:54:38 WARN DAGScheduler: Broadcasting large task binary with size 1008.8 KiB
26/06/03 20:54:38 WARN DAGScheduler: Broadcasting large task binary with size 1009.2 KiB
26/06/03 20:54:38 WARN DAGScheduler: Broadcasting large task binary with size 1010.2 KiB
26/06/03 20:54:38 WARN DAGScheduler: Broadcasting large task binary with size 1011.0 KiB
26/06/03 20:54:38 WARN DAGScheduler: Broadcasting large task binary with size 1013.3 KiB
26/06/03 20:54:38 WARN DAGScheduler: Broadcasting large task binary with size 1015.9 KiB
26/06/03 20:54:39 WAR


Best GBT parameters:
  cacheNodeIds: False
  checkpointInterval: 10
  featureSubsetStrategy: all
  featuresCol: raw_features
  impurity: variance
  labelCol: log_Value
  leafCol: 
  lossType: squared
  maxBins: 32
  maxDepth: 4
  maxIter: 150
  maxMemoryInMB: 256
  minInfoGain: 0.0
  minInstancesPerNode: 1
  minWeightFractionPerNode: 0.0
  predictionCol: prediction
  seed: 42
  stepSize: 0.2
  subsamplingRate: 1.0
  validationTol: 0.01


In [19]:
preds_gbt_tuned = evaluate_model("GBT (tuned)", cv_model_gbt, test_raw)


Model : GBT (tuned)
  R²              : 0.9957
  RMSE (log-scale): 0.0814
  MAE  (log-scale): 0.0575
  MAE  (Euros €)  : 207,179.22 €


<a class="anchor" id="7"></a>

# **7. Model Comparison**

[Back to TOC](#toc)

In [20]:
results_df = pd.DataFrame(results).T.reset_index().rename(columns={"index": "Model"})
print(results_df.to_string(index=False))

                       Model     R²  RMSE (log)  MAE (log)   MAE (€)
Linear Regression (baseline) 0.9660      0.2291     0.1644 607781.60
    Random Forest (baseline) 0.9883      0.1346     0.0951 344961.98
              GBT (baseline) 0.9952      0.0859     0.0593 214294.66
       Random Forest (tuned) 0.9925      0.1078     0.0746 275777.74
                 GBT (tuned) 0.9957      0.0814     0.0575 207179.22


In [21]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=["RMSE (log-scale) ↓", "MAE (log-scale) ↓", "R² ↑"]
)

colors = px.colors.qualitative.Set2[:len(results_df)]

for metric, col_idx in [("RMSE (log)", 1), ("MAE (log)", 2), ("R²", 3)]:
    fig.add_trace(
        go.Bar(
            x=results_df["Model"],
            y=results_df[metric],
            name=metric,
            marker_color=colors,
            text=results_df[metric].round(4),
            textposition="outside"
        ),
        row=1, col=col_idx
    )

fig.update_layout(
    title="Model Comparison — Test Set Metrics",
    showlegend=False,
    height=450
)
fig.update_xaxes(tickangle=-30)
fig.show()

In [22]:
# Predicted vs Actual scatter for best model (GBT tuned)
best_preds = preds_gbt_tuned.select("log_Value", "prediction").toPandas()
# NOTE: .toPandas() used here only for visualisation on a small result set.
# In production with truly large data, this would be replaced by writing
# the predictions to distributed storage and sampling.

fig = px.scatter(
    best_preds.sample(min(3000, len(best_preds)), random_state=42),
    x="log_Value", y="prediction",
    opacity=0.4,
    labels={"log_Value": "Actual log(Value)", "prediction": "Predicted log(Value)"},
    title="GBT (tuned) — Predicted vs Actual (log scale)"
)

min_v = best_preds["log_Value"].min()
max_v = best_preds["log_Value"].max()
fig.add_trace(go.Scatter(
    x=[min_v, max_v], y=[min_v, max_v],
    mode="lines", name="Perfect",
    line=dict(color="red", dash="dash")
))
fig.show()

In [23]:
# Residual plot
best_preds["residual"] = best_preds["prediction"] - best_preds["log_Value"]

fig = px.scatter(
    best_preds.sample(min(3000, len(best_preds)), random_state=42),
    x="prediction", y="residual",
    opacity=0.4,
    labels={"prediction": "Predicted log(Value)", "residual": "Residual"},
    title="GBT (tuned) — Residuals vs Predicted"
)
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.show()

<a class="anchor" id="8"></a>

# **8. Overfitting Analysis (Train vs Test)**

[Back to TOC](#toc)

In [29]:
trained_models = {
    "Linear Reg":  model_lr,
    "RF (Base)":   model_rf,
    "GBT (Base)":  model_gbt,
    "RF (Tuned)":  cv_model_rf.bestModel,
    "GBT (Tuned)": cv_model_gbt.bestModel
}

overfit_data = []

for name, model in trained_models.items():
    preds_train = model.transform(train_raw)
    rmse_train  = evaluator_rmse.evaluate(preds_train)

    preds_test  = model.transform(test_raw)
    rmse_test   = evaluator_rmse.evaluate(preds_test)

    overfit_data.append({"Model": name, "Dataset": "Train", "RMSE": rmse_train})
    overfit_data.append({"Model": name, "Dataset": "Test",  "RMSE": rmse_test})

overfit_df = pd.DataFrame(overfit_data)

fig = px.bar(
    overfit_df,
    x="Model", y="RMSE", color="Dataset",
    barmode="group",
    title="Overfitting Analysis: Train vs Test RMSE (log-scale)",
    labels={"RMSE": "RMSE (Lower is better)"},
    color_discrete_map={"Train": "royalblue", "Test": "darkorange"},
    text="RMSE"
)
fig.update_traces(
    texttemplate='%{text:.4f}', textposition='outside',
    marker_line_color='black', marker_line_width=1
)
fig.update_layout(height=500, template="plotly_white")
fig.show()

<a class="anchor" id="9"></a>

# **9. Feature Importance**

[Back to TOC](#toc)

Random Forest and GBT both expose `featureImportances`. We use the tuned RF model here as it provides stable importance estimates across trees.

In [30]:
best_rf_model = cv_model_rf.bestModel.stages[-1]  # RandomForestRegressionModel

# Reconstruct feature names after OHE
ohe_feature_names = []
for c in categorical_features:
    # The OHE stage index within the pipeline
    ohe_stage_idx = len(
        [sql_clean, ContractFeatureEngineer(), NPositionsEngineer(), LogTargetTransformer()]
    ) + len(indexers) + categorical_features.index(c)
    ohe_model = cv_model_rf.bestModel.stages[ohe_stage_idx]
    n_cats = ohe_model.categorySizes[0] - 1
    ohe_feature_names += [f"{c}_{i}" for i in range(n_cats)]

feature_names = imputed_numeric + ohe_feature_names

importances = best_rf_model.featureImportances.toArray()
min_len = min(len(feature_names), len(importances))

feat_imp_df = pd.DataFrame({
    "Feature":    feature_names[:min_len],
    "Importance": importances[:min_len]
}).sort_values("Importance", ascending=False).head(25)

fig = px.bar(
    feat_imp_df,
    x="Importance", y="Feature",
    orientation="h",
    title="Top 25 Feature Importances — Random Forest (tuned)",
    color="Importance", color_continuous_scale="Blues",
    text="Importance"
)
fig.update_traces(texttemplate='%{text:.4f}', textposition='outside')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
fig.show()

In [31]:
# GBT feature importances
best_gbt_model = cv_model_gbt.bestModel.stages[-1]

gbt_importances = best_gbt_model.featureImportances.toArray()
min_len_gbt = min(len(feature_names), len(gbt_importances))

gbt_imp_df = pd.DataFrame({
    "Feature":    feature_names[:min_len_gbt],
    "Importance": gbt_importances[:min_len_gbt]
}).sort_values("Importance", ascending=False).head(25)

fig = px.bar(
    gbt_imp_df,
    x="Importance", y="Feature",
    orientation="h",
    title="Top 25 Feature Importances — GBT (tuned)",
    color="Importance", color_continuous_scale="Greens",
    text="Importance"
)
fig.update_traces(texttemplate='%{text:.4f}', textposition='outside')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
fig.show()

In [32]:
# Final summary table
print("\n" + "="*60)
print("FINAL MODEL COMPARISON")
print("="*60)
print(results_df.sort_values("R²", ascending=False).to_string(index=False))
print("="*60)
print("Note: all metrics computed on log1p(Value). Lower RMSE/MAE and higher R² is better.")


FINAL MODEL COMPARISON
                       Model     R²  RMSE (log)  MAE (log)   MAE (€)
                 GBT (tuned) 0.9957      0.0814     0.0575 207179.22
              GBT (baseline) 0.9952      0.0859     0.0593 214294.66
       Random Forest (tuned) 0.9925      0.1078     0.0746 275777.74
    Random Forest (baseline) 0.9883      0.1346     0.0951 344961.98
Linear Regression (baseline) 0.9660      0.2291     0.1644 607781.60
Note: all metrics computed on log1p(Value). Lower RMSE/MAE and higher R² is better.
